In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from mintpy.utils import utils as ut
from scipy.stats import pearsonr

In [ ]:
def load_snotel_data_pickle(filename="snotel_data.pkl"):
    """Loads SNOTEL data dictionary from a pickle file."""
    with open(filename, "rb") as f:
        return pickle.load(f)

def extract_site_coordinates(swe_data):
    """
    Extracts site names and their latitude/longitude from the results dictionary.

    Args:
        results (dict): Dictionary where keys are site names and values are Pandas DataFrames.

    Returns:
        pd.DataFrame: DataFrame containing site names, latitude, and longitude.
    """
    site_data = []

    for site_name, df in swe_data.items():
        # Extract the first non-null site_loc (assuming all rows have the same location)
        if not df.empty and "site_loc" in df.columns:
            point = df["site_loc"].iloc[0]  # This is a Shapely Point object
            lon, lat = point.x, point.y  # Extract coordinates

            # Store extracted info
            site_data.append({"site_name": site_name, "latitude": lat, "longitude": lon})

    # Convert to DataFrame for easy handling
    return pd.DataFrame(site_data)

def filter_with_dates(swe_data, dates):
    filtered_data = swe_data[swe_data["date_time_utc"].dt.date.isin(dates)].copy()

    return filtered_data

def get_delta_swe_snotel(swe_data, site_loc_list, dates):
    station_swe = swe_data[site_loc_list.iloc[0].site_name]
    filtered_station_swe = filter_with_dates(station_swe, dates.astype("datetime64[D]"))
    filtered_data = filtered_station_swe["value_cm"].diff()
    filtered_data.iloc[0] = 0

def get_displacement_data(lat, lon, ts_file, lookup_file):
    """Reads displacement time series data for a given location."""
    dates, dis, dis_std = ut.read_timeseries_lalo(
        lat=lat, lon=lon, ts_file=ts_file, lookup_file=str(lookup_file)
    )
    return np.array([d.date() for d in dates]), dis

def process_swe_data(swe_data, site_name, insar_dates):
    """Filters SWE data and computes daily changes."""
    filtered_swe = filter_with_dates(swe_data[site_name], insar_dates)
    filtered_data = filtered_swe["value_cm"].diff()
    filtered_data.iloc[0] = 0  # Set first value to 0
    return filtered_data

def compute_displacement_changes(dis):
    """Computes daily changes in displacement."""
    delta_dis = np.diff(dis)
    return np.insert(delta_dis, 0, 0)  # Insert 0 at the beginning

def plot_displacements(insar_dates, vh_delta_dis, vv_delta_dis, filtered_data):
    """Plots displacement data with cumulative sums."""
    fig, ax = plt.subplots(figsize=[10, 5])

    ax.scatter(insar_dates, np.cumsum(vh_delta_dis * 100), marker="^", label="VH")
    ax.scatter(insar_dates, np.cumsum(vv_delta_dis * 100), marker="o", label="VV")
    ax.scatter(insar_dates, np.cumsum(filtered_data), marker="*", label="InSitu")

    # Axis formatting
    ax.tick_params(which="both", bottom=True, left=True)
    ax.set_xlabel("Time [year]")
    ax.set_ylabel("LOS displacement [cm]")
    ax.legend(loc="upper left", fontsize=10, frameon=True)

    fig.tight_layout()
    plt.show()

def calculate_swe(ts_arr, inc_angle):
    # ki = 2*pi/wvl # Just for calculating the ambiguity
    F = (-0.6784*inc_angle**2)+(0.2899*inc_angle)-0.8473
    return ts_arr/-F

## Load SWE data from the previous notebook

In [ ]:
swe_data = load_snotel_data_pickle("snotel_P071_F444.pkl")

In [ ]:
site_loc_list = extract_site_coordinates(swe_data)

## Load InSAR timeseries

In [ ]:
timeseries_dir = Path('/Users/soveisgh/Desktop/PC_Desktop/Snow/Sentinel/data/USA/2021/Path_071_Frame_444/mintpy').expanduser()
timeseries_file = timeseries_dir.joinpath('timeseries_ERA5.h5')
#lookup_file = timeseries_dir.joinpath('inputs/geometryRadar.h5')
lookup_file = timeseries_dir.joinpath('hyp3_incidence_angle.tif') 



In [ ]:
for idx in range(1): #range(len(site_loc_list)):
    dates, dis, dis_std = ut.read_timeseries_lalo(
        lat=site_loc_list.iloc[idx].latitude,
        lon=site_loc_list.iloc[idx].longitude,
        win_size=10,
        #ref_lat=39.163952,
        #ref_lon=-119.896721,
        ts_file=str(timeseries_file),
        # lookup_file=str(lookup_file),
    )

    insar_dates = np.array([d.date() for d in dates])
    station_swe = swe_data[site_loc_list.iloc[idx].site_name]

    filtered_station_swe = filter_with_dates(station_swe, insar_dates)
    filtered_data = filtered_station_swe["value_cm"].diff()
    filtered_data.iloc[0] = 0.0  # start from zero change
    filtered_data = filtered_data
    
    inc_angle, _ = ut.readfile.read(lookup_file, datasetName='incidenceAngle')
    atr = ut.readfile.read_attribute(str(timeseries_file))
    coord = ut.coordinate(atr, lookup_file=str(lookup_file))
    coord.open()
    y, x = coord.geo2radar(site_loc_list.iloc[idx].latitude, site_loc_list.iloc[idx].longitude)[0:2]
    inc_angle = np.deg2rad(inc_angle[y, x])

    delta_dis = np.diff(-dis)
    insar_swe = calculate_swe(delta_dis, inc_angle)  # likely meters
    insar_swe = np.insert(insar_swe, 0, 0.0)         # align lengths
    insar_swe_cm = (insar_swe) * 100.0                 # convert to cm
    insar_swe_cm = insar_swe_cm
    insar_dates = insar_dates

    # ---- Metrics (mask NaNs and require at least 2 points) ----
    y_true = np.asarray(filtered_data.values, dtype=float)
    y_pred = np.asarray(insar_swe_cm, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() >= 2:
        resid = y_pred[mask] - y_true[mask]
        rmse = np.sqrt(np.mean(resid**2))
        var = np.var(y_true[mask], ddof=0)
        r2 = 1.0 - np.sum(resid**2) / np.sum((y_true[mask] - y_true[mask].mean())**2) if var > 0 else np.nan
    else:
        rmse, r2 = np.nan, np.nan

    cor_delswe, p_value = pearsonr(insar_swe_cm, filtered_data)
    
    cumulative_insar_swe_cm = np.cumsum(insar_swe_cm)
    cumulative_filtered_data = np.cumsum(filtered_data)

    cor_swe, p_value = pearsonr(cumulative_insar_swe_cm, cumulative_filtered_data)

    # ---- Plot ----
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=[10, 5])
    ax.plot(insar_dates, insar_swe_cm, marker="^", label="InSAR")
    ax.plot(insar_dates, filtered_data, marker="o", label="InSitu")

    # axis format
    ax.tick_params(which='both', direction='in', bottom=True, top=True, left=True, right=True)
    ax.set_xlabel('Time [year]')
    ax.set_ylabel(r'$\Delta$SWE [cm]')
    ax.legend()
    ax.set_title(site_loc_list.iloc[idx].site_name)

    # annotate metrics on the plot
    txt = f"RMSE = {rmse:.2f} cm\n$R^2$ = {r2:.2f}"
    ax.text(0.02, 0.98, txt, transform=ax.transAxes, ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.75))

    fig.tight_layout()
    plt.show()

    # ---- Plot ----
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=[10, 5])
    ax.plot(insar_dates, cumulative_insar_swe_cm, marker="^", label="InSAR")
    ax.plot(insar_dates, cumulative_filtered_data, marker="o", label="InSitu")

    # axis format
    ax.tick_params(which='both', direction='in', bottom=True, top=True, left=True, right=True)
    ax.set_xlabel('Time [year]')
    ax.set_ylabel('SWE [cm]')
    ax.legend()
    ax.set_title(site_loc_list.iloc[idx].site_name)

    